Finnhub: d4m5t5hr01qjidhtok10d4m5t5hr01qjidhtok1g

In [2]:
!pip install finnhub

ERROR: Could not find a version that satisfies the requirement finnhub (from versions: none)
ERROR: No matching distribution found for finnhub


In [7]:
import os, time, hashlib
from datetime import datetime, timedelta
import pandas as pd, numpy as np
import finnhub, requests, yfinance as yf
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from pandas.tseries.offsets import BDay
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [11]:
# ------------------- CONFIG -------------------

# FINNHUB_API_KEY = "d2arrjpr01qgk9ugd2ugd2arrjpr01qgk9ugd2v0"
# MARKETAUX_API_KEY = "3Ej6emn1jO1MzOjziTsrCacnbQvx6pjueugwiMb4"
SYMBOL = "NVDA"
CUTOVER_DATE = datetime(2024, 8, 28).date()  # use Finnhub on or after
YEAR_START = "2023-07-01"
YEAR_END = "2025-08-01"
LAG_DAYS = [1, 3, 7, 14]
OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# API endpoints
MARKETAUX_URL = "https://api.marketaux.com/v1/news/all"

# ------------------- INIT CLIENT -------------------

# finnhub_client = finnhub.Client(api_key=FINNHUB_API_KEY)

# Building Data

In [ ]:
# ------------------- 1) FETCH NEWS -------------------

all_news = []
start = datetime.strptime(YEAR_START, "%Y-%m-%d")
end = datetime.strptime(YEAR_END, "%Y-%m-%d")
delta = timedelta(days=7)

print("Fetching NVDA news using Marketaux (before cutover) and Finnhub (after)...")

while start <= end:
    batch_end = min(start + delta, end)
    date_from, date_to = start.date(), batch_end.date()

    #  if before cutoff date — use Marketaux
    if date_to < CUTOVER_DATE:
        # Marketaux request
        params = {
            "api_token": MARKETAUX_API_KEY,
            "symbols": SYMBOL,
            "published_after": f"{date_from}T00:00:00",
            "published_before": f"{date_to}T23:59:59",
            "limit": 1000
        }
        try:
            resp = requests.get(MARKETAUX_URL, params=params, timeout=10)
            resp.raise_for_status()
            data = resp.json().get("data", [])
            print(f"Marketaux: {len(data)} articles {date_from}→{date_to}")
            for a in data:
                all_news.append({
                    "id": a["uuid"],
                    "timestamp": a["published_at"],
                    "headline": a["title"],
                    "summary": a["description"],
                    "url": a["url"],
                    "source": "marketaux"
                })
        except Exception as e:
            print(f"Marketaux error {date_from}→{date_to}: {e}")

    #  if on/after cutoff date — use Finnhub
    else:
        try:
            fh = finnhub_client.company_news(SYMBOL, _from=str(date_from), to=str(date_to))
            print(f"Finnhub: {len(fh)} articles {date_from}→{date_to}")
            for a in fh:
                all_news.append({
                    "id": a["id"],
                    "timestamp": pd.to_datetime(a["datetime"], unit="s", utc=True),
                    "headline": a["headline"],
                    "summary": a.get("summary", ""),
                    "url": a.get("url", ""),
                    "source": "finnhub"
                })
        except Exception as e:
            print(f"Finnhub error {date_from}→{date_to}: {e}")

    start = batch_end + timedelta(days=1)
    time.sleep(0.3)

# ------------------- Deduplicate & Normalize -------------------

df = pd.DataFrame(all_news)
# hash headline+summary+timestamp
df["ts_str"] = df["timestamp"].astype(str)
df["hash"] = (df["headline"].fillna("") + df["summary"].fillna("") + df["ts_str"]).apply(
    lambda x: hashlib.sha1(x.encode()).hexdigest()
)
df = df.drop_duplicates(subset=["hash"]).reset_index(drop=True)
df["published_at"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["published_at"])
df["pub_date"] = df["published_at"].dt.date
df["text"] = (df["headline"] + ". " + df["summary"]).str.strip()

print(f"Total news after deduplication: {len(df)}")
print("Date range:", df['pub_date'].min(), "→", df['pub_date'].max())

df.to_csv(os.path.join(OUT_DIR, "news_combined.csv"), index=False)

In [ ]:
# ------------------- 2) Sentiment Scoring -------------------

print("Scoring sentiment (FinBERT)...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(DEVICE)

def score(text):
    if not text:
        return (0,0,1,0)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model(**inputs)
    neg, neu, pos = softmax(out.logits.cpu().numpy()[0])
    return pos-neg, pos, neu, neg

sent = df["text"].apply(lambda t: pd.Series(score(t), index=["sent_net","sent_pos","sent_neu","sent_neg"]))
df = pd.concat([df, sent], axis=1)
df.to_csv(os.path.join(OUT_DIR, "news_with_sentiment.csv"), index=False)
print("Sentiment scoring done.")

In [ ]:
# ------------------- 3) Fetch Prices -------------------

start_dt = (pd.to_datetime(YEAR_START) - timedelta(days=5)).strftime("%Y-%m-%d")
end_price = df["pub_date"].max() + BDay(max(LAG_DAYS)+5)
print("Fetching historical prices:", start_dt, "→", end_price.date())
prices = yf.download(SYMBOL, start=start_dt, end=end_price.strftime("%Y-%m-%d"), interval="1d")
prices.reset_index(inplace=True)
prices["date"] = pd.to_datetime(prices["Date"]).dt.date
prices = prices[["date","Open","High","Low","Close","Volume"]]
prices.to_csv(os.path.join(OUT_DIR, "price_data.csv"), index=False)
print("Price data saved.")

# Next: compute lags, directional labels, correlation, etc. (unchanged)

In [6]:
# -------------------------
# 4) Compute lagged pct changes per news article
# -------------------------

# Use price DataFrame saved earlier
df_price = prices.copy()
df_price["date"] = pd.to_datetime(df_price["date"])
df_price = df_price.sort_values("date").reset_index(drop=True)
trading_dates = list(sorted(df_price["date"].unique()))

def get_trading_date_on_or_after(input_date):
    input_date = pd.Timestamp(input_date)
    for d in trading_dates:
        if d >= input_date:
            return d
    return None

def get_close_on_date(date):
    match = df_price.loc[df_price["date"] == date, "Close"]
    if len(match) == 0:
        return None
    return float(match.iloc[0])

def add_trading_days(start_date, days):
    if start_date not in trading_dates:
        start_date = get_trading_date_on_or_after(start_date)
    if start_date is None:
        return None
    try:
        start_idx = trading_dates.index(start_date)
    except ValueError:
        return None
    target_idx = start_idx + days
    if target_idx >= len(trading_dates):
        return None
    return trading_dates[target_idx]

records = []
price_min_date = df_price["date"].min()
price_max_date = df_price["date"].max()
print(f"Price data covers from {price_min_date.date()} to {price_max_date.date()}")

for _, r in df.iterrows():
    pub_date = pd.to_datetime(r["pub_date"])
    if pub_date < price_min_date:
        print(f"Warning: News date {pub_date.date()} before price data start {price_min_date.date()}. Skipping.")
        continue
    if pub_date > price_max_date:
        print(f"Warning: News date {pub_date.date()} after price data end {price_max_date.date()}. Skipping.")
        continue

    base_td = get_trading_date_on_or_after(pub_date)
    if base_td is None:
        print(f"Warning: No trading date found on/after news date {pub_date.date()}. Skipping.")
        continue
    base_close = get_close_on_date(base_td)
    if base_close is None:
        print(f"Warning: No close price for base trading date {base_td.date()}. Skipping.")
        continue

    rec = {
        "id": r.get("id", None),
        "published_at": r.get("published_at", None),
        "pub_date": pub_date,
        "base_trading_date": base_td,
        "base_close": base_close,
        "sent_net": r["sent_net"],
        "sent_pos": r["sent_pos"],
        "sent_neu": r["sent_neu"],
        "sent_neg": r["sent_neg"],
        "headline": r.get("headline", ""),
        "url": r.get("url", ""),
        "source": r.get("source", "")
    }

    for lag in LAG_DAYS:
        target_date = add_trading_days(base_td, lag)
        if target_date is None:
            print(f"Note: Lag {lag}d out of range for article on {pub_date.date()}.")
            rec[f"close_{lag}d"] = np.nan
            rec[f"pct_change_{lag}d"] = np.nan
            rec[f"trading_date_{lag}d"] = None
            continue

        close_price = get_close_on_date(target_date)
        if close_price is None:
            print(f"Warning: No price for lag {lag}d date {target_date.date()} (article {pub_date.date()}).")
            rec[f"close_{lag}d"] = np.nan
            rec[f"pct_change_{lag}d"] = np.nan
            rec[f"trading_date_{lag}d"] = None
            continue

        rec[f"close_{lag}d"] = close_price
        rec[f"pct_change_{lag}d"] = (close_price - base_close) / base_close
        rec[f"trading_date_{lag}d"] = target_date

    records.append(rec)

df_results = pd.DataFrame(records)
df_results.to_csv(os.path.join(OUT_DIR, "nvda_news_lagged_results.csv"), index=False)
print(f"Saved per-article lagged results. Processed articles: {len(df_results)}")

NameError: name 'prices' is not defined

In [29]:
# ================================
# Standalone Step 4: recompute lagged pct changes
# using existing price_data.csv and news_with_sentiment.csv
# ================================
import os
import numpy as np
import pandas as pd
from datetime import datetime

OUT_DIR = "./outputs"
LAG_DAYS = [1, 3, 7, 14, 30]

price_path = os.path.join(OUT_DIR, "price_data.csv")
news_sent_path = os.path.join(OUT_DIR, "news_with_sentiment.csv")

df_price = pd.read_csv(price_path)
df_price["date"] = pd.to_datetime(df_price["date"])
df_price = df_price.sort_values("date").reset_index(drop=True)

df = pd.read_csv(news_sent_path)
df["pub_date"] = pd.to_datetime(df["pub_date"])

trading_dates = list(sorted(df_price["date"].unique()))

def get_trading_date_on_or_after(input_date):
    input_date = pd.Timestamp(input_date)
    for d in trading_dates:
        if d >= input_date:
            return d
    return None

def get_close_on_date(date):
    match = df_price.loc[df_price["date"] == date, "Close"]
    if len(match) == 0:
        return None
    return float(match.iloc[0])

def add_trading_days(start_date, days):
    if start_date not in trading_dates:
        start_date = get_trading_date_on_or_after(start_date)
    if start_date is None:
        return None
    try:
        start_idx = trading_dates.index(start_date)
    except ValueError:
        return None
    target_idx = start_idx + days
    if target_idx >= len(trading_dates):
        return None
    return trading_dates[target_idx]

records = []
price_min_date = df_price["date"].min()
price_max_date = df_price["date"].max()
print(f"Price data covers from {price_min_date.date()} to {price_max_date.date()}")

for _, r in df.iterrows():
    pub_date = pd.to_datetime(r["pub_date"])
    if pub_date < price_min_date or pub_date > price_max_date:
        continue

    base_td = get_trading_date_on_or_after(pub_date)
    if base_td is None:
        continue
    base_close = get_close_on_date(base_td)
    if base_close is None:
        continue

    rec = {
        "id": r.get("id", None),
        "published_at": r.get("published_at", None),
        "pub_date": pub_date,
        "base_trading_date": base_td,
        "base_close": base_close,
        "sent_net": r["sent_net"],
        "sent_pos": r["sent_pos"],
        "sent_neu": r["sent_neu"],
        "sent_neg": r["sent_neg"],
        "headline": r.get("headline", ""),
        "url": r.get("url", ""),
        "source": r.get("source", "")
    }

    for lag in LAG_DAYS:
        target_date = add_trading_days(base_td, lag)
        if target_date is None:
            rec[f"close_{lag}d"] = np.nan
            rec[f"pct_change_{lag}d"] = np.nan
            rec[f"trading_date_{lag}d"] = None
            continue

        close_price = get_close_on_date(target_date)
        if close_price is None:
            rec[f"close_{lag}d"] = np.nan
            rec[f"pct_change_{lag}d"] = np.nan
            rec[f"trading_date_{lag}d"] = None
            continue

        rec[f"close_{lag}d"] = close_price
        rec[f"pct_change_{lag}d"] = (close_price - base_close) / base_close
        rec[f"trading_date_{lag}d"] = target_date

    records.append(rec)

df_results = pd.DataFrame(records)
out_path = os.path.join(OUT_DIR, "nvda_news_lagged_results.csv")
df_results.to_csv(out_path, index=False)
print(f"Saved per-article lagged results -> {out_path}, Processed articles: {len(df_results)}")

Price data covers from 2023-06-26 to 2025-09-15
Saved per-article lagged results -> ./outputs/nvda_news_lagged_results.csv, Processed articles: 8779


# Analysis

In [124]:
# ================================
# Standalone Step 5: Analysis using existing lag file
# (no need to rerun steps 1–3 if CSVs already exist)
# ================================
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression

OUT_DIR = "./outputs"

# --- Load precomputed lagged results ---
lag_file = os.path.join(OUT_DIR, "nvda_news_lagged_results.csv")
if not os.path.exists(lag_file):
    raise FileNotFoundError(
        f"{lag_file} not found. You must run steps 1–4 at least once to create it."
    )

df_results = pd.read_csv(lag_file)

# Optional: ensure numeric types
df_results["sent_net"] = pd.to_numeric(df_results["sent_net"], errors="coerce")
for lag in LAG_DAYS:
    col = f"pct_change_{lag}d"
    if col in df_results.columns:
        df_results[col] = pd.to_numeric(df_results[col], errors="coerce")

print("Analyzing results by lag...")
summary_rows = []

for lag in LAG_DAYS:
    col = f"pct_change_{lag}d"

    if "sent_net" not in df_results.columns or col not in df_results.columns:
        print(f"Skipping lag {lag}d — missing 'sent_net' or '{col}' in df_results.")
        continue

    tmp = df_results[["sent_net", col]].dropna()
    n = len(tmp)

    if n >= 5:
        r_val, p_val = stats.pearsonr(tmp["sent_net"], tmp[col])
        X = tmp[["sent_net"]].values.reshape(-1, 1)
        y = tmp[col].values
        lr = LinearRegression().fit(X, y)
        coef = float(lr.coef_[0])
        intercept = float(lr.intercept_)
        r2 = float(lr.score(X, y))
    else:
        r_val = p_val = coef = intercept = r2 = np.nan

    tmp_dir = tmp.assign(
        sent_dir=np.sign(tmp["sent_net"]),
        price_dir=np.sign(tmp[col])
    )
    tmp_dir = tmp_dir[tmp_dir["sent_dir"] != 0]
    accuracy = (tmp_dir["sent_dir"] == tmp_dir["price_dir"]).mean() if len(tmp_dir) > 0 else np.nan

    summary_rows.append({
        "lag_days": lag,
        "n": n,
        "pearson_r": r_val,
        "p_value": p_val,
        "reg_coef": coef,
        "reg_intercept": intercept,
        "reg_r2": r2,
        "directional_accuracy": accuracy
    })

df_summary = pd.DataFrame(summary_rows).sort_values("lag_days")
if not df_summary.empty:
    summary_csv = os.path.join(OUT_DIR, "nvda_lag_summary.csv")
    df_summary.to_csv(summary_csv, index=False)
    print("Saved lag summary ->", summary_csv)
    print(df_summary)
else:
    print("No valid lag results to save.")

Analyzing results by lag...
Saved lag summary -> ./outputs/nvda_lag_summary.csv
   lag_days     n  pearson_r   p_value  reg_coef  reg_intercept    reg_r2  \
0         1  8779  -0.017746  0.096380 -0.000903       0.002384  0.000315   
1         3  8779   0.008597  0.420572  0.000767       0.008689  0.000074   
2         7  8779   0.005073  0.634634  0.000565       0.026118  0.000026   
3        14  8779  -0.020580  0.053826 -0.003234       0.036015  0.000424   

   directional_accuracy  
0              0.500854  
1              0.504499  
2              0.549493  
3              0.526825  


In [125]:
# -------------------------
# 6) Compute per-article best lag
# -------------------------
def best_lag_info(row):
    vals = row[[f"pct_change_{lag}d" for lag in LAG_DAYS]].abs()
    vals = vals.dropna()
    if vals.empty:
        return (None, np.nan)
    best_col = vals.idxmax()
    best_lag = int(best_col.split("_")[2].replace("d",""))
    best_val = row[best_col]
    return (best_lag, best_val)

df_results["best_lag_days"] = df_results.apply(lambda r: best_lag_info(r)[0], axis=1)
df_results["best_lag_pct_change"] = df_results.apply(lambda r: best_lag_info(r)[1], axis=1)
df_results.to_csv(os.path.join(OUT_DIR, "nvda_news_lagged_results_with_bestlag.csv"), index=False)
print("Saved per-article best-lag file.")

Saved per-article best-lag file.


In [126]:
# -------------------------
# 7) Plots
# -------------------------
plt.figure(figsize=(8,5))
plt.plot(df_summary["lag_days"], df_summary["pearson_r"], marker="o")
plt.xlabel("Lag after news (days)")
plt.ylabel("Pearson correlation (sent_net vs pct_change)")
plt.title("NVDA: sentiment vs pct change correlation by lag")
plt.grid(True)
plt.savefig(os.path.join(OUT_DIR, "correlation_vs_lag.png"), dpi=200)
plt.close()

plt.figure(figsize=(8,5))
plt.bar(df_summary["lag_days"].astype(str), df_summary["directional_accuracy"])
plt.xlabel("Lag (days)")
plt.ylabel("Directional accuracy (fraction)")
plt.title("Direction accuracy of sentiment prediction by lag")
plt.grid(axis="y")
plt.savefig(os.path.join(OUT_DIR, "directional_accuracy_by_lag.png"), dpi=200)
plt.close()

print("Plots saved in", OUT_DIR)
print("Done.")

Plots saved in ./outputs
Done.


In [127]:
# --------- load prerequisites from disk (so we don't need steps 1–3) ---------
df_results = pd.read_csv(os.path.join(OUT_DIR, "nvda_news_lagged_results.csv"))
df_price   = pd.read_csv(os.path.join(OUT_DIR, "price_data.csv"))

In [ ]:
# ================================
# 8) FEATURE ENGINEERING (standalone daily panel)
# ================================
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# --- Load from saved CSVs (no need to rerun steps 1–3) ---
price_file = os.path.join(OUT_DIR, "price_data.csv")
lags_file  = os.path.join(OUT_DIR, "nvda_news_lagged_results.csv")

df_price = pd.read_csv(price_file)
df_results = pd.read_csv(lags_file)

# --- Basic cleaning / typing ---
# Dates
df_price["date"] = pd.to_datetime(df_price["date"])
df_results["pub_date"] = pd.to_datetime(df_results["pub_date"])

# Numeric price columns
price_num_cols = ["Open", "High", "Low", "Close", "Volume"]
for c in price_num_cols:
    if c in df_price.columns:
        df_price[c] = pd.to_numeric(df_price[c], errors="coerce")

# Numeric sentiment columns (if present)
for c in ["sent_net", "sent_pos", "sent_neu", "sent_neg"]:
    if c in df_results.columns:
        df_results[c] = pd.to_numeric(df_results[c], errors="coerce")

# --- aggregate daily sentiment from articles ---
df_results["pub_date"] = df_results["pub_date"].dt.normalize()

news_day = (
    df_results
    .groupby("pub_date")
    .agg(
        sent_net_mean=("sent_net", "mean"),
        sent_net_sum=("sent_net", "sum"),
        sent_pos_mean=("sent_pos", "mean"),
        sent_neg_mean=("sent_neg", "mean"),
        n_articles=("id", "count")
    )
    .sort_index()
    .reset_index()
)

# --- price features ---
px = df_price.copy()
px["date"] = px["date"].dt.normalize()
px = px.sort_values("date").rename(columns={"Close": "close"}).set_index("date")

# returns & vol
px["ret_1d"]  = px["close"].pct_change(1)
px["ret_5d"]  = px["close"].pct_change(5)
px["ret_10d"] = px["close"].pct_change(10)
px["vol_10d"] = px["ret_1d"].rolling(10).std()
px["vol_20d"] = px["ret_1d"].rolling(20).std()

# flatten multiindex columns in px (defensive)
if isinstance(px.columns, pd.MultiIndex):
    px.columns = ["_".join([str(c) for c in col]).strip("_") for col in px.columns]

# ensure news_day index is 'date'
news_day = news_day.rename(columns={"pub_date": "date"}).set_index("date")

# merge features
feat = px.join(news_day, how="left")

# fill no-news days with zeros (no news signal that day)
for c in ["sent_net_mean", "sent_net_sum", "sent_pos_mean", "sent_neg_mean", "n_articles"]:
    if c in feat.columns:
        feat[c] = feat[c].fillna(0)


# ================================
# 🔹 NEW STEP 2: extra predictive features
# ================================

# auto-detect close column (handles "close" or "close_NVDA" cases)
close_candidates = [c for c in feat.columns if c.startswith("close")]
if len(close_candidates) == 0:
    raise ValueError(f"No 'close*' column found in feat.columns: {feat.columns.tolist()}")
close_col = close_candidates[0]
print(f"Using close column: {close_col}")

# 1) Price momentum features
feat["mom_3d"]  = feat[close_col].pct_change(3)
feat["mom_7d"]  = feat[close_col].pct_change(7)
feat["mom_14d"] = feat[close_col].pct_change(14)

# 2) Volatility regime (ratio of long vs short vol)
feat["vol_regime"] = feat["vol_20d"] / feat["vol_10d"]

# 3) Sentiment momentum (rolling averages of daily sentiment)
feat["sent_mom_3d"] = feat["sent_net_mean"].rolling(3, min_periods=1).mean()
feat["sent_mom_7d"] = feat["sent_net_mean"].rolling(7, min_periods=1).mean()


# --- future label horizons ---
HORIZONS = [1, 3, 5, 10]  # trading days ahead

for h in HORIZONS:
    # return over next h days
    feat[f"label_ret_{h}d"] = feat[close_col].shift(-h) / feat[close_col] - 1.0
    
    # 3-class direction label: -1, 0, +1 (with neutral band)
    band = 0.002  # +/- 0.2% as neutral band
    lab = feat[f"label_ret_{h}d"].apply(
        lambda r: 1 if r > band else (-1 if r < -band else 0)
    )
    feat[f"label_dir_{h}d"] = lab

# Drop rows with NaNs in any of the features / labels
feat = feat.dropna().copy()

print("Feature panel 'feat' ready. Shape:", feat.shape)
print("Columns:", feat.columns.tolist())

Using close column: close
Feature panel 'feat' ready. Shape: (528, 29)
Columns: ['Open', 'High', 'Low', 'close', 'Volume', 'ret_1d', 'ret_5d', 'ret_10d', 'vol_10d', 'vol_20d', 'sent_net_mean', 'sent_net_sum', 'sent_pos_mean', 'sent_neg_mean', 'n_articles', 'mom_3d', 'mom_7d', 'mom_14d', 'vol_regime', 'sent_mom_3d', 'sent_mom_7d', 'label_ret_1d', 'label_dir_1d', 'label_ret_3d', 'label_dir_3d', 'label_ret_5d', 'label_dir_5d', 'label_ret_10d', 'label_dir_10d']


In [ ]:
# ================================
# 8b) Simple binary labels (no neutral band)
# ================================

# 1-day ahead binary label: 1 if next-day return > 0, else 0
feat["y_bin"] = (feat["label_ret_1d"] > 0).astype(int)

print("Binary 1-day label created: y_bin")
print(feat["y_bin"].value_counts())
print("\nClass distribution (%):")
print(feat["y_bin"].value_counts(normalize=True).round(3))

Binary 1-day label created: y_bin
y_bin
1    292
0    236
Name: count, dtype: int64

Class distribution (%):
y_bin
1    0.553
0    0.447
Name: proportion, dtype: float64


In [ ]:
# ================================
# 10) Feature matrix X and target y (H = 1d)
# ================================

# Choose features: price history + volatility + sentiment + news intensity
feature_cols = [
    "ret_1d", "ret_5d", "ret_10d",
    "vol_10d", "vol_20d",
    "mom_3d", "mom_7d", "mom_14d",
    "vol_regime",
    "sent_net_mean", "sent_net_sum",
    "sent_pos_mean", "sent_neg_mean",
    "sent_mom_3d", "sent_mom_7d",
    "n_articles",
]

# Ensure all required columns exist in feat
missing = [c for c in feature_cols if c not in feat.columns]
if missing:
    raise ValueError(f"Missing feature columns in feat: {missing}")

# Drop rows with NaNs in features or target
model_df = feat.dropna(subset=feature_cols + ["y_bin"]).copy()

print("Final modeling dataset shape:", model_df.shape)

# Design X and y (keep X as DataFrame so index = dates)
X = model_df[feature_cols]
y = model_df["y_bin"]

# Simple time-based split: first 80% train, last 20% test
n = len(model_df)
split_idx = int(n * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train size: {len(y_train)}, Test size: {len(y_test)}")
print("Train class distribution:", y_train.value_counts().to_dict())
print("Test class distribution:", y_test.value_counts().to_dict())

Final modeling dataset shape: (528, 30)
Train size: 422, Test size: 106
Train class distribution: {1: 231, 0: 191}
Test class distribution: {1: 61, 0: 45}


In [136]:
# ================================
# 12) Binary Direction (H=1d) — Random Forest baseline
# ================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]   # <-- will use this for threshold tuning

acc_rf  = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf  = recall_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf)
auc_rf  = roc_auc_score(y_test, y_proba_rf)

print("=== Binary Direction (H=1d) — Random Forest (default threshold=0.5) ===")
print(f"Accuracy : {acc_rf:.3f}")
print(f"Precision: {prec_rf:.3f}")
print(f"Recall   : {rec_rf:.3f}")
print(f"F1 score : {f1_rf:.3f}")
print(f"ROC AUC  : {auc_rf:.3f}")
print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification report:")
print(classification_report(y_test, y_pred_rf, digits=3))


# ==========================================
# 12b) STEP 3 — tune decision threshold
# ==========================================

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print("\n=== Threshold sweep (positive=UP trade) ===")
best_thr = None
best_f1  = -1

for thr in thresholds:
    y_pred_thr = (y_proba_rf >= thr).astype(int)

    acc  = accuracy_score(y_test, y_pred_thr)
    prec = precision_score(y_test, y_pred_thr, zero_division=0)
    rec  = recall_score(y_test, y_pred_thr, zero_division=0)
    f1   = f1_score(y_test, y_pred_thr, zero_division=0)

    print(f"thr={thr:.2f}  |  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}")

    if f1 > best_f1:
        best_f1  = f1
        best_thr = thr

print(f"\nBest threshold by F1: {best_thr:.2f} (F1={best_f1:.3f})")

# Optional: inspect confusion matrix at best_thr
y_pred_best = (y_proba_rf >= best_thr).astype(int)
print("\nConfusion matrix at best_thr (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred_best))

=== Binary Direction (H=1d) — Random Forest (default threshold=0.5) ===
Accuracy : 0.462
Precision: 0.532
Recall   : 0.541
F1 score : 0.537
ROC AUC  : 0.427

Confusion matrix (rows=true, cols=pred):
[[16 29]
 [28 33]]

Classification report:
              precision    recall  f1-score   support

           0      0.364     0.356     0.360        45
           1      0.532     0.541     0.537        61

    accuracy                          0.462       106
   macro avg      0.448     0.448     0.448       106
weighted avg      0.461     0.462     0.461       106


=== Threshold sweep (positive=UP trade) ===
thr=0.30  |  acc=0.575  prec=0.575  rec=1.000  f1=0.731
thr=0.40  |  acc=0.575  prec=0.582  rec=0.934  f1=0.717
thr=0.50  |  acc=0.462  prec=0.532  rec=0.541  f1=0.537
thr=0.60  |  acc=0.415  prec=0.462  rec=0.098  f1=0.162
thr=0.70  |  acc=0.425  prec=0.000  rec=0.000  f1=0.000

Best threshold by F1: 0.30 (F1=0.731)

Confusion matrix at best_thr (rows=true, cols=pred):
[[ 0 45]
 [ 0

In [137]:
# ================================
# 12c) STEP 4 — Long / Short / Flat 1-day Strategy (H=1d)
# ================================
import numpy as np
import pandas as pd

# 1) Get the test dates from X_test (we kept the index in cell 10)
test_idx = X_test.index

# 2) Realized next-day returns for those test dates
#    This was created in feature-engineering: label_ret_1d
realized_ret_1d = feat.loc[test_idx, "label_ret_1d"]

# 3) Convert RF probabilities to a Series indexed by test dates
proba_up = pd.Series(y_proba_rf, index=test_idx)

# 4) Define upper and lower thresholds for "confident" signals
#    You can tweak these, but this is a reasonable starting point:
upper = 0.60   # only go LONG if P(UP) >= 60%
lower = 0.40   # only go SHORT if P(UP) <= 40%

# 5) Build trading signal:
#    +1 = long, -1 = short, 0 = flat
signal = pd.Series(0, index=test_idx)
signal[proba_up >= upper] = 1
signal[proba_up <= lower] = -1

# 6) Strategy daily returns: position * next-day return
strategy_ret = signal * realized_ret_1d

# --- Basic stats ---
n_days    = len(test_idx)
n_long    = (signal == 1).sum()
n_short   = (signal == -1).sum()
n_flat    = (signal == 0).sum()

long_ret  = strategy_ret[signal == 1]
short_ret = strategy_ret[signal == -1]

avg_long_ret  = long_ret.mean() if len(long_ret) > 0 else np.nan
avg_short_ret = short_ret.mean() if len(short_ret) > 0 else np.nan

win_rate_long  = (long_ret > 0).mean() if len(long_ret) > 0 else np.nan
win_rate_short = (short_ret > 0).mean() if len(short_ret) > 0 else np.nan  # "win" = profit on short

# Total cumulative returns
cum_strategy = (1 + strategy_ret).prod() - 1
cum_buyhold  = (1 + realized_ret_1d).prod() - 1  # always long benchmark

print("=== 1-day Long/Short/Flat Strategy (test set) ===")
print(f"Horizon: 1 trading day")
print(f"Upper threshold (long): {upper:.2f}")
print(f"Lower threshold (short): {lower:.2f}")
print(f"Number of test days: {n_days}")
print(f"Days long : {n_long}")
print(f"Days short: {n_short}")
print(f"Days flat : {n_flat}\n")

if n_long > 0:
    print(f"Avg return when LONG : {avg_long_ret*100:.3f}%")
    print(f"Win rate when LONG  : {win_rate_long*100:.1f}%")
else:
    print("No LONG trades taken.")

if n_short > 0:
    print(f"\nAvg return when SHORT: {avg_short_ret*100:.3f}%")
    print(f"Win rate when SHORT : {win_rate_short*100:.1f}%")
else:
    print("\nNo SHORT trades taken.")

print(f"\nCumulative strategy return (test period): {cum_strategy*100:.2f}%")
print(f"Cumulative buy-and-hold over same period: {cum_buyhold*100:.2f}%")

=== 1-day Long/Short/Flat Strategy (test set) ===
Horizon: 1 trading day
Upper threshold (long): 0.60
Lower threshold (short): 0.40
Number of test days: 106
Days long : 13
Days short: 8
Days flat : 85

Avg return when LONG : 0.739%
Win rate when LONG  : 46.2%

Avg return when SHORT: -0.658%
Win rate when SHORT : 50.0%

Cumulative strategy return (test period): 1.53%
Cumulative buy-and-hold over same period: 57.59%
